# Instituto Tecnológico de Ciudad Madero
## Departamento de Sistemas y Computación - Ingeniería en Sistemas Computacionales
### Asignatura: Tópicos Avanzados de Programación
**Docente:** Alejandro Estrada Padilla  
**Práctica:** Análisis y visualización de calificaciones con Pandas y Matplotlib

## Ejercicio 1. Carga y revisión
> **¿Por qué el archivo tiene 288 registros y no 24?**  
> El conjunto contiene 24 alumnos en total. Cada alumno cursa 4 asignaturas distintas (Programacion, Bases de datos, Redes y Matematicas discretas), y cada una de ellas se evalúa en 3 unidades individuales.  
> Por ende: **24 alumnos × 4 materias × 3 unidades = 288 registros**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Cargar el archivo CSV
df = pd.read_csv("calificaciones_isc.csv")

print("--- 1. Primeras 5 filas ---")
display(df.head())

print("\n--- 2. Dimensiones (Filas y Columnas) ---")
print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}")

print("\n--- 3. Tipo de dato de cada columna ---")
print(df.dtypes)

print("\n--- 4. Resumen estadístico de calificacion y de asistencias ---")
display(df[["calificacion", "asistencias"]].describe())

## Ejercicio 2. Filtros
Filtros booleanos sin modificar `df` y contando con `len()` (sin usar ciclos).

In [ ]:
# 1. Solo el grupo A
filtro_grupo_a = df[df["grupo"] == "A"]
print(f"Filas correspondientes solo al Grupo A: {len(filtro_grupo_a)}")

# 2. Unidades con calificación menor a 70
filtro_menor_70 = df[df["calificacion"] < 70]
print(f"Filas con calificación menor a 70 (< 70): {len(filtro_menor_70)}")

# 3. Bases de datos, del grupo B, con calificación >= 80
filtro_bd_b_80 = df[(df["materia"] == "Bases de datos") & (df["grupo"] == "B") & (df["calificacion"] >= 80)]
print(f"Filas de Bases de datos, Grupo B, calificación >= 80: {len(filtro_bd_b_80)}")

## Ejercicio 3. Columna calculada (`situacion`)
Se asigna "Aprobado" si la calificación es $\ge 70$, de lo contrario "Reprobado", usando `np.where`.

In [ ]:
df["situacion"] = np.where(df["calificacion"] >= 70, "Aprobado", "Reprobado")
display(df[["matricula", "nombre", "materia", "unidad", "calificacion", "situacion"]].head(10))

## Ejercicio 4. Resúmenes y Exportación a CSV

In [ ]:
# 1. Promedio de calificación por materia
promedio_materia = df.groupby("materia")["calificacion"].mean()
print("--- 1. Promedio por materia ---")
print(promedio_materia.round(2))

# 2. Promedio por grupo y materia
promedio_grupo_materia = df.groupby(["grupo", "materia"])["calificacion"].mean()
print("\n--- 2. Promedio por grupo y materia ---")
print(promedio_grupo_materia.round(2))

# 3. Promedio general de cada alumno, ordenado de mayor a menor
promedios_alumnos = df.groupby(["matricula", "nombre"])["calificacion"].mean().reset_index()
promedios_alumnos = promedios_alumnos.rename(columns={"calificacion": "promedio"})
promedios_alumnos = promedios_alumnos.sort_values(by="promedio", ascending=False)
print("\n--- 3. Promedio general de cada alumno (Top 10) ---")
display(promedios_alumnos.head(10))

# 4. Promedio general por unidad
promedio_unidad = df.groupby("unidad")["calificacion"].mean()
print("\n--- 4. Promedio general por unidad ---")
print(promedio_unidad.round(2))

# 5. Conteo de unidades aprobadas y reprobadas
conteo_situacion = df["situacion"].value_counts()
print("\n--- 5. Unidades aprobadas vs reprobadas ---")
print(conteo_situacion)

# Exportación a CSV sin guardar índice
promedios_alumnos.to_csv("promedios_alumnos.csv", index=False)
print("\n[OK] promedios_alumnos.csv exportado exitosamente sin índice.")

## Ejercicio 5. Gráficas con Matplotlib
Se guardan como PNG antes de mostrarse.

In [ ]:
# Figura 1: Barras del promedio por materia (Eje Y: 0 a 100)
plt.figure(figsize=(8, 5))
colores = ["#3498db", "#2ecc71", "#e67e22", "#9b59b6"]
barras = plt.bar(promedio_materia.index, promedio_materia.values, color=colores, edgecolor="black", width=0.55)
plt.title("Promedio de Calificación por Materia", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Materia", fontsize=12, fontweight="bold")
plt.ylabel("Calificación Promedio", fontsize=12, fontweight="bold")
plt.ylim(0, 100)
plt.grid(axis="y", linestyle="--", alpha=0.7)
for b in barras:
    h = b.get_height()
    plt.text(b.get_x() + b.get_width() / 2.0, h + 1.5, f"{h:.2f}", ha="center", va="bottom", fontweight="bold")
plt.tight_layout()
plt.savefig("promedio_materias.png", dpi=300)
plt.show()

# Figura 2: Histograma con línea vertical en 70
plt.figure(figsize=(8, 5))
plt.hist(df["calificacion"], bins=12, color="#5dade2", edgecolor="black", alpha=0.8)
plt.axvline(x=70, color="red", linestyle="--", linewidth=2.5, label="Calificación Aprobatoria (70)")
plt.title("Distribución General de Calificaciones", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Calificación Obtenida", fontsize=12, fontweight="bold")
plt.ylabel("Frecuencia (Unidades Evaluadas)", fontsize=12, fontweight="bold")
plt.legend(loc="upper left")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.savefig("histograma.png", dpi=300)
plt.show()

# Figura 3: Líneas de promedio por unidad por materia
tendencia = df.groupby(["unidad", "materia"])["calificacion"].mean().unstack()
plt.figure(figsize=(9, 5.5))
marcadores = ["o", "s", "^", "D"]
for idx, m in enumerate(tendencia.columns):
    plt.plot(tendencia.index, tendencia[m], marker=marcadores[idx % len(marcadores)], linewidth=2.2, markersize=8, label=m)
plt.title("Tendencia del Promedio por Unidad según Materia", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Unidad de Evaluación", fontsize=12, fontweight="bold")
plt.ylabel("Calificación Promedio", fontsize=12, fontweight="bold")
plt.xticks([1, 2, 3], ["Unidad 1", "Unidad 2", "Unidad 3"], fontsize=11)
plt.grid(True, linestyle="--", alpha=0.7)
plt.legend(title="Materias", loc="best")
plt.tight_layout()
plt.savefig("tendencia_unidades.png", dpi=300)
plt.show()

## Ejercicio 6. Lectura de Resultados
1. **¿Qué materia quedó con el promedio más bajo?**  
   **Matemáticas discretas** (promedio: **66.08**), siendo la única por debajo de 70.
2. **¿Es el promedio general de la unidad 3 menor que el de las unidades 1 y 2?**  
   **Sí**. El promedio de la **Unidad 3 es 69.79**, menor que el de la Unidad 1 (**73.75**) y Unidad 2 (**73.16**).
3. **¿Cuántos alumnos quedan bajo 70 en su promedio general?**  
   Un total de **7 alumnos** quedan con promedio acumulado inferior a 70.